RELEVANT NOTES:

Black-Scholes:

Call : $C(S,t) = SN(d_1) - Ke^{-r(T-t)}N(d_2) $   , Put : $P(S,t) = Ke^{-r(T-t)}N(-d_2) - SN(-d_1) $

where, $d_1 = \frac{ln(S/K) + (r + \sigma^2/2)(T-t)}{\sigma\sqrt{T-t}}$       ,       $d_2 = d_1 - \sigma\sqrt{T-t}$, N is the distribution function of $\phi(0,1)$

in our case the current time 't' is 0 so we can just use 'T'

Binomial Tree:

time step $\Delta t = \frac{T}{\text{number of time steps(n)}}$ in our case, T = 0.75 and n = 274 after rounding up 

p = $\frac{e^{rT}-d}{u-d}$

from Delta and Volatility: u = $e^{\sigma \sqrt{\Delta t}}$ and d = $e^{-\sigma \sqrt{\Delta t}}$

from risk neutral probabilities and valuation: f = $\frac{pf_u +(1-p)f_d}{e^{rT}}$ where, $f$ - option price today, $f_u$ - option price if the stock goes up, $f_d$ - option price if stock goes down, $e^{-rT}$ is the discount factor so we can use that to go from right to left and fill in the nodes of the tree one time step at a time

we need the values for $f_u$ and $f_d$ so we need the stock price at every step, we will be since from Generalisations(1) calculating that through S = $S_0 * u^i x d^{steps-i}$ where i is the number of times stock goes up over the course of the steps that are taken 

from valuing american options, the process will be to work back through the tree from end to the beginning, same as european but we need an extra step to test if early exercise will give better value at final node, value of american option should be same as european, at earlier, it is the greater of the “European” value computed from $e^{-rT}[pf_u + (1-p)f_d]$ or, the payoff from early exercise (i.e., the value at the step itself)

Monte-Carlo:

Equation for stock prices is broken up into two parts:
Certain Part: $\frac{dS}{S} = \mu dt$ and the uncertain part where the uncertainty is gaussian and proportional to $\sqrt{\Delta t}$ translating to ....+$\sigma \epsilon \sqrt{dt}$ leading us to the model $dS = \mu Sdt + \sigma Sdz$ where $\sigma$ is the volatility of the stock price and $\mu$ is the expected return (drift coefficient)

from handout6, price = $e^{-rT}\hat{E}[payoff(S_T)]$
path simulation equation: $S(t+\Delta t) = S(t)\Delta t + \sigma S(t)\epsilon\sqrt{\Delta t}$ and from handout 4, we have $\epsilon \textasciitilde \phi(0,1)$

from handout6, it can be more accurate to simulate lnS through the equation: $dlnS = (r-\sigma^2/2)dt + \sigma dz$ so its still certain part + uncertain part
so we obtain lnS and then get S from $e^{lnS}$

In [3]:
import math
import random 
from scipy.stats import norm

#Inputs:

X1 = 120 #strike price 1
X2 = 96 #from largest and second largest number in student ID strike price 2
#K will be strike price
T = 0.75 #time to maturity 3/4 of a year
S0 = 100 #current stock price
sigma = 0.3 #annual volatility
r = 0.05 #annual interest rate (risk free)
steps = round(365*T) #so each time step is roughly rounded to 1 day 
paths = 100000 # might change depending on runtime, higher is more accurate


def bs(S0, K, r, sigma, T, option_type):
    d1 = ( math.log(S0/K) + (r+0.5*sigma**2)*T)/(sigma * math.sqrt(T)) 
    d2 = d1 - sigma *math.sqrt(T)

    if option_type == "call":
        return S0 * norm.cdf(d1) - K * math.exp(-r*T) * norm.cdf(d2)
    else:
        return K * math.exp(-r*T) * norm.cdf(-d2) - S0 * norm.cdf(-d1)

def trees(S0,K,r,sigma,T,steps, option_type, region):
    
    dt = T/steps
    u = math.exp(sigma * math.sqrt(dt))
    d = 1/u #since its basically $u^{-1}$
    p =(math.exp(r*dt)-d)/(u-d) 
    discount_factor = math.exp(-r * dt)
    values = []

    #to figure out the payoff when the option matures:
    for i in range(steps+1):
        S=S0 * (u**i)*(d**(steps-i)) # obtain the stock price at each step
        if option_type =="call":
            values.append(max(S-K,0)) 
        else:
            values.append(max(K-S,0))

    for m in reversed(range(steps)):   #iterating backwards from the final step to the first basically from right to left of tree
        updated_values =[]
        for i in range (m+1):  #since there will be m + 1 nodes at m
            
            continuation_value = discount_factor*(p*values[i+1]+(1-p)*values[i]) 
            # essentially if i is the number of moves up a tree, then values[i+1] means the option value at the upper node and values[i]
            # is the value at the lower node

            if region == "american":
                S=S0*(u**i)*(d**(m-i))       #calculating the stock price at the current node
                if option_type == "call":
                    value_at_step = max(S-K,0)  
                else:
                    value_at_step = max(K-S,0)
                updated_values.append(max(continuation_value,value_at_step))

            else:
                updated_values.append(continuation_value)

        values = updated_values
    return values[0]

def montecarlo(S0 , K , r , sigma , T , steps , paths , option_type ):
    
    dt = T/steps
    payoff_total = 0

    for i in range(paths):     #loops through paths
        logS = math.log(S0)
        for m in range(steps):     #loops through steps in each path
            epsilon = random.gauss(0,1)  #randomizing the path simulation
            logS += (r-0.5*sigma**2)*dt + sigma*epsilon*math.sqrt(dt)
        S = math.exp(logS)
        if option_type == "call":
            payoff_total += max(S-K,0)
        else:
            payoff_total += max(K-S,0)
            
    price = math.exp(-r*T)*(payoff_total/paths)
    return price

def results(K):
    
    print("\n Strike Price = ",K)
    
    print("\n Black-Scholes ")
    print("European Call = ", bs(S0, K, r, sigma, T, "call"))
    print("European Put = ", bs(S0, K, r, sigma, T, "put"))
    
    print("\n Binomial Trees ")
    print("European Call = ", trees(S0,K,r,sigma,T,steps, "call","european"))
    print("European Put = ", trees(S0,K,r,sigma,T,steps, "put","european"))
    print("American Call = ", trees(S0,K,r,sigma,T,steps, "call","american"))
    print("American Put = ", trees(S0,K,r,sigma,T,steps, "put","american"))
    
    print("\n Monte Carlo ")
    print("European Call = ", montecarlo(S0, K, r, sigma, T, steps, paths, "call"))
    print("European Put = ", montecarlo(S0, K, r, sigma, T, steps, paths, "put"))
    
#passing values for x1 and x2 
results(X1)
results(X2)


 Strike Price =  120

 Black-Scholes 
European Call =  5.0236808576470295
European Put =  20.607010984145646

 Binomial Trees 
European Call =  5.0236790423847815
European Put =  20.607009168883607
American Call =  5.0236790423847815
American Put =  21.927198754046557

 Monte Carlo 
European Call =  5.004646553681031
European Put =  20.54860420118083

 Strike Price =  96

 Black-Scholes 
European Call =  14.158868804789847
European Put =  6.6255329059887345

 Binomial Trees 
European Call =  14.164503040654726
European Put =  6.631167141853641
American Call =  14.164503040654726
American Put =  6.8980457152516665

 Monte Carlo 
European Call =  14.203019431681904
European Put =  6.62463093851266
